In [19]:
import os
import glob
import argparse
import numpy as np
import torch
import rasterio
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from torchgeo.models import croma_base, CROMABase_Weights

In [20]:
TILES_BASE = os.path.expanduser('~/Desktop/Master/thesis/tiles/croma')

### Optical experiment 1

In [21]:
device = torch.device('mps')  # or 'cpu'
                                                                                                                            
model = croma_base(
      weights=CROMABase_Weights.CROMA_VIT,                                                                                  
      modalities=['optical'],
  )                                                                                                                         
model = model.to(device)
if hasattr(model, 'attn_bias') and model.attn_bias is not None:                                                           
      model.attn_bias = model.attn_bias.to(device)                                                                          
model.eval()
print("Model loaded") 

Model loaded


In [22]:
records = []
for fid_dir in sorted(glob.glob(os.path.join(TILES_BASE, 'fid_*'))):
      fid = os.path.basename(fid_dir)
      for category in ['s2_evt', 's2_bef']:
          cat_dir = os.path.join(fid_dir, category)
          if not os.path.isdir(cat_dir):
              continue
          evt_bef = 'evt' if 'evt' in category else 'bef'
          for image_dir in sorted(glob.glob(os.path.join(cat_dir, '*'))):
              image_id = os.path.basename(image_dir)
              for tile_path in sorted(glob.glob(os.path.join(image_dir, '*.tif'))):
                  records.append({
                      'fid': fid,
                      'evt_bef': evt_bef,
                      's2_path': tile_path,
                      's1_path': None,
                      'image_id': image_id,
                  })

print(f"Total S2 tiles: {len(records)}")
print(f"  evt: {sum(1 for r in records if r['evt_bef'] == 'evt')}")
print(f"  bef: {sum(1 for r in records if r['evt_bef'] == 'bef')}")

Total S2 tiles: 381
  evt: 55
  bef: 326


In [23]:
def load_tile(path):
    """Load a GeoTIFF tile as a numpy array (C, H, W)."""
    real_path = os.path.realpath(path)
    with rasterio.open(real_path) as ds:
        data = ds.read()  # (C, H, W)
    return data.astype(np.float32)

In [24]:
def normalize(x):
    """Per-channel robust normalization to [0, 1]."""
    x = x.float()
    imgs = []
    for ch in range(x.shape[1]):
        channel = x[:, ch, :, :]
        min_val = channel.mean() - 2 * channel.std()
        max_val = channel.mean() + 2 * channel.std()
        img = (channel - min_val) / (max_val - min_val + 1e-10)
        img = torch.clamp(img, 0, 1)
        imgs.append(img.unsqueeze(1))
    return torch.cat(imgs, dim=1)

In [25]:
embeddings = []
fids = []
evt_befs = []

for start in range(0, len(records), 32):
      batch = records[start:start+32]
      tensors = [torch.from_numpy(load_tile(r['s2_path'])) for r in batch]
      x = normalize(torch.stack(tensors).to(device))

      with torch.no_grad():
          out = model(x_sar=None, x_optical=x)

      embeddings.append(out['optical_GAP'].cpu().numpy())
      fids.extend([r['fid'] for r in batch])
      evt_befs.extend([r['evt_bef'] for r in batch])

embeddings = np.concatenate(embeddings, axis=0)
fids = np.array(fids)
evt_befs = np.array(evt_befs)
print(f"Embeddings shape: {embeddings.shape}")  # (N, 768)

Embeddings shape: (381, 768)


In [26]:
# Basic stats
print(f"Mean: {embeddings.mean():.4f}, Std: {embeddings.std():.4f}")
print(f"Min: {embeddings.min():.4f}, Max: {embeddings.max():.4f}")

  # Per-class stats
for eb in ['evt', 'bef']:
    mask = evt_befs == eb
    print(f"\n{eb} ({mask.sum()} tiles):")
    print(f"  Mean norm: {np.linalg.norm(embeddings[mask], axis=1).mean():.2f}")

Mean: -0.0129, Std: 0.4087
Min: -1.8850, Max: 1.8486

evt (55 tiles):
  Mean norm: 11.47

bef (326 tiles):
  Mean norm: 11.15


In [27]:
from rasterio.transform import from_bounds

RGB_DIR = os.path.expanduser('~/Desktop/Master/thesis/tiles/rgb_previews')
os.makedirs(RGB_DIR, exist_ok=True)

def make_rgb_preview(s2_path, output_path):
      """Create an RGB PNG from a 12-band CROMA tile (B4=Red, B3=Green, B2=Blue)."""
      if os.path.exists(output_path):
          return output_path

      with rasterio.open(s2_path) as ds:
          data = ds.read().astype(np.float32)

      # CROMA band order: B1,B2,B3,B4,B5,B6,B7,B8,B8A,B9,B11,B12
      # B4=index 3 (Red), B3=index 2 (Green), B2=index 1 (Blue)
      rgb = data[[3, 2, 1], :, :]  # (3, 120, 120)

      # Normalize to 0-255 (per-channel mean ± 2*std)
      for ch in range(3):
          band = rgb[ch]
          vmin = band.mean() - 2 * band.std()
          vmax = band.mean() + 2 * band.std()
          rgb[ch] = np.clip((band - vmin) / (vmax - vmin + 1e-10) * 255, 0, 255)

      rgb = rgb.astype(np.uint8)

      os.makedirs(os.path.dirname(output_path), exist_ok=True)
      # Save as PNG using matplotlib
      plt.imsave(output_path, np.transpose(rgb, (1, 2, 0)))
      return output_path


In [28]:
import fiftyone as fo                                                                               
import fiftyone.brain as fob                                                                      
                                                                                                      
  # Create dataset
dataset = fo.Dataset("croma_deforestation", overwrite=True)                                         
                                                                                                    
samples = []
for i, rec in enumerate(records):
      # Generate RGB preview
      tile_name = os.path.basename(rec['s2_path']).replace('.tif', '.png')
      rgb_path = os.path.join(RGB_DIR, rec['fid'], rec['evt_bef'], rec['image_id'], tile_name)
      make_rgb_preview(rec['s2_path'], rgb_path)

      sample = fo.Sample(filepath=rgb_path)  # <-- use RGB, not the GeoTIFF
      sample["fid"] = rec['fid']
      sample["evt_bef"] = rec['evt_bef']
      sample["image_id"] = rec['image_id']
      samples.append(sample)

dataset.add_samples(samples)
print(dataset)

 100% |█████████████████| 381/381 [42.0ms elapsed, 0s remaining, 9.1K samples/s]   
Name:        croma_deforestation
Media type:  image
Num samples: 381
Persistent:  False
Tags:        []
Sample fields:
    id:               fiftyone.core.fields.ObjectIdField
    filepath:         fiftyone.core.fields.StringField
    tags:             fiftyone.core.fields.ListField(fiftyone.core.fields.StringField)
    metadata:         fiftyone.core.fields.EmbeddedDocumentField(fiftyone.core.metadata.ImageMetadata)
    created_at:       fiftyone.core.fields.DateTimeField
    last_modified_at: fiftyone.core.fields.DateTimeField
    fid:              fiftyone.core.fields.StringField
    evt_bef:          fiftyone.core.fields.StringField
    image_id:         fiftyone.core.fields.StringField


### Attach the CROMA embeddings and compute UMAP in FiftyOne

In [29]:
# Attach the 768-d embeddings to each sample                                                        
for i, sample in enumerate(dataset):
      sample["croma_embedding"] = embeddings[i].tolist()                                              
      sample.save()
                                                                                                      
# Let FiftyOne compute the 2D visualization (UMAP internally)                                       
fob.compute_visualization(                                                                          
      dataset,                                                                                        
      embeddings=embeddings,  # (N, 768) numpy array
      brain_key="croma_umap",
      method="umap",
      seed=42,
  )               

Generating visualization...


/opt/anaconda3/envs/geo_latest/lib/python3.11/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


UMAP(n_jobs=1, random_state=42, verbose=True)
Thu Mar 26 16:38:33 2026 Construct fuzzy simplicial set
Thu Mar 26 16:38:33 2026 Finding Nearest Neighbors
Thu Mar 26 16:38:33 2026 Finished Nearest Neighbor Search
Thu Mar 26 16:38:33 2026 Construct embedding


Epochs completed:  16%| █▋         82/500 [00:00]

	completed  0  /  500 epochs
	completed  50  /  500 epochs
	completed  100  /  500 epochs


...

	completed  250  /  500 epochs
	completed  300  /  500 epochs


Epochs completed:  82%| ████████▏  411/500 [00:00]

	completed  350  /  500 epochs
	completed  400  /  500 epochs
	completed  450  /  500 epochs


Epochs completed: 100%| ██████████ 500/500 [00:00]

Thu Mar 26 16:38:34 2026 Finished embedding


In [30]:
session = fo.launch_app(dataset)

### Conclusions of my first CROMA attemp
My model can not distiguish between deforestation/ no deforestation.
They more or less keep the representations of the same FID in the same region. 